<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Notebook 3 — SQL Analytics : Performance & Attribution
### 📝 VERSION APPRENANT

> Objectif : Répondre aux questions business avec du SQL avancé — RANK(), LAG(), NTILE().
> Chaque requête a une question métier à résoudre.

---
## 0. Setup

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

pd.set_option('display.max_columns', 50)
COLORS = {'primary':'#534AB7','secondary':'#1D9E75','warning':'#EF9F27','danger':'#E24B4A','neutral':'#888780','light':'#EEEDFE'}
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#F9F9F8','axes.grid':True,'grid.alpha':0.35,'font.size':11})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'
print(f'Configuration OK — {SAVE_PATH}')


In [ ]:
import duckdb
%load_ext sql
conn = duckdb.connect()
analytics_path = f'{SAVE_PATH}marketpulse_analytics.csv'
if os.path.exists(analytics_path):
    conn.execute(f"CREATE TABLE marketpulse_analytics AS SELECT * FROM read_csv_auto('{analytics_path}', timestampformat='%Y-%m-%d')")
else:
    conn.execute(f"CREATE TABLE marketpulse_analytics AS SELECT * FROM read_csv_auto('{BASE_URL}marketpulse_analytics.csv', timestampformat='%Y-%m-%d')")
conn.execute(f"CREATE TABLE performances_daily AS SELECT * FROM read_csv_auto('{BASE_URL}performances_daily.csv', timestampformat='%Y-%m-%d');")
conn.execute(f"CREATE TABLE campagnes AS SELECT * FROM read_csv_auto('{BASE_URL}campagnes.csv', timestampformat='%Y-%m-%d');")
conn.execute(f"CREATE TABLE clients_agence AS SELECT * FROM read_csv_auto('{BASE_URL}clients_agence.csv', timestampformat='%Y-%m-%d');")
conn.execute(f"CREATE TABLE contacts_crm AS SELECT * FROM read_csv_auto('{BASE_URL}contacts_crm.csv', timestampformat='%Y-%m-%d');")
conn.execute(f"CREATE TABLE audiences AS SELECT * FROM read_csv_auto('{BASE_URL}audiences.csv');")
conn.execute("ALTER TABLE performances_daily ADD COLUMN IF NOT EXISTS ctr_recalc DOUBLE;")
conn.execute("ALTER TABLE performances_daily ADD COLUMN IF NOT EXISTS roas_recalc DOUBLE;")
conn.execute("UPDATE performances_daily SET ctr_recalc = CASE WHEN impressions>0 THEN ROUND(clics*1.0/impressions,4) END, roas_recalc = CASE WHEN depenses_fcfa>0 AND revenu_genere_fcfa>=0 THEN ROUND(revenu_genere_fcfa*1.0/depenses_fcfa,3) END")
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('DuckDB + JupySQL prêts ✅')


---
## 2. Vue consolidée depuis `marketpulse_analytics`

### MÉTHODE
Première requête sur la table analytique du NB2. Utilise `marketpulse_analytics` (source de vérité campagne-niveau) pour les KPIs globaux.

In [ ]:
%%sql df_kpi_global <<
-- 📝 TODO : Calculer les KPIs globaux : nb_campagnes, nb_clients, ROAS moyen, médiane, CTR%, CPA, CPC, nb_sous_perf, % sous-perf, ROAS_J3 moyen, durée moy
-- 💡 Indice 1 : PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY roas_total) AS roas_median
-- 💡 Indice 2 : COUNT(DISTINCT client_id) AS nb_clients
-- 💡 Indice 3 : ROUND(SUM(campagne_sous_performe)*100.0/COUNT(*),1) AS pct_sous_perf
-- 💡 Indice 4 : FROM marketpulse_analytics


### 🧠 Tes observations

> - Le ROAS médian est-il proche du ROAS moyen ? Qu'indique l'écart entre les deux ?
> - Le ROAS J3 moyen est-il plus bas que le ROAS final ? Quelle est la logique d'optimisation des algorithmes publicitaires ?


---
## 3. RANK() — Classement des canaux

### MÉTHODE
Un seul passage suffit pour calculer plusieurs classements simultanés. CTE pour les agrégats, requête externe pour les RANK().

In [ ]:
%%sql df_rank_canal <<
-- 📝 TODO : Calculer ROAS, CTR, CPC, CPA, % sous-perf par canal + 4 RANK() simultanés + score composite
-- 💡 Indice 1 : WITH perf_canal AS (SELECT canal, COUNT(*), AVG(roas_total)... FROM marketpulse_analytics GROUP BY canal)
-- 💡 Indice 2 : RANK() OVER (ORDER BY roas_moyen DESC) AS rang_roas
-- 💡 Indice 3 : RANK() OVER (ORDER BY cpc_fcfa ASC) AS rang_cout
-- 💡 Indice 4 : RANK() OVER (ORDER BY pct_sous_perf ASC) AS rang_fiabilite
-- 💡 Indice 5 : Score composite = sum des 3 rangs (meilleur = 3)


### 🧠 Tes observations

> - Quel canal a le meilleur score composite ? Correspond-il à tes attentes du NB1 ?
> - Pourquoi SMS a-t-il un CPC si bas comparé à Google ? (Pense au modèle économique de chaque canal)


---
## 4. RANK() PARTITION BY — Canal × Objectif

### MÉTHODE
`RANK() OVER (PARTITION BY objectif ORDER BY roas DESC)` classe les canaux **au sein de chaque objectif** — donne une recommandation plus précise qu'un classement global.

In [ ]:
%%sql df_canal_objectif <<
-- 📝 TODO : Croiser canal × objectif avec RANK() dans chaque objectif
-- 💡 Indice 1 : WITH croise AS (SELECT canal, objectif_campagne, COUNT(*), AVG(roas_total)... HAVING COUNT(*)>=3)
-- 💡 Indice 2 : RANK() OVER (PARTITION BY objectif_campagne ORDER BY roas_moyen DESC) AS rang_dans_objectif


### 🧠 Tes observations

> - Pour l'objectif 'Conversion' : quel canal est recommandé ? Pour 'Notoriété' ?
> - Cette analyse change-t-elle tes recommandations du tableau global RANK() ?


---
## 5. LAG() — Tendance mensuelle

### MÉTHODE
`LAG(roas) OVER (ORDER BY mois)` retourne la valeur du mois précédent. `LAG(roas, 2)` retourne celle d'il y a 2 mois — détecte les tendances baissières continues.

In [ ]:
%%sql df_lag_mensuel <<
-- 📝 TODO : Tendance mensuelle : ROAS, CTR, delta_roas avec LAG() + alerte 2 mois consécutifs
-- 💡 Indice 1 : strftime(date_debut, '%Y-%m') AS mois
-- 💡 Indice 2 : LAG(roas) OVER (ORDER BY mois) AS roas_prev
-- 💡 Indice 3 : roas - LAG(roas) AS delta_roas
-- 💡 Indice 4 : CASE WHEN roas < LAG(roas) AND LAG(roas) < LAG(roas,2) THEN '⚠️ Déclin 2 mois' END


### 🧠 Tes observations

> - Combien de mois avec `⚠️ Déclin 2 mois` as-tu identifié ? Sur quelles périodes ?
> - Quel est le mois avec la plus forte hausse de ROAS ? Quelle logique marketing l'explique ?


---
## 6. NTILE(3) — Segmentation campagnes

### MÉTHODE
`NTILE(3) OVER (ORDER BY roas_total DESC)` divise en 3 tiers égaux. Calculer le profil de chaque tier identifie les patterns qui distinguent les meilleures campagnes.

In [ ]:
%%sql df_ntile_camp <<
-- 📝 TODO : Segmenter les 354 campagnes en 3 tiers avec NTILE(3) + profil par tier
-- 💡 Indice 1 : NTILE(3) OVER (ORDER BY roas_total DESC) AS tier_roas
-- 💡 Indice 2 : CASE tier_roas WHEN 1 THEN 'Top performer' WHEN 2 THEN 'Intermédiaire' ELSE 'À optimiser' END AS segment
-- 💡 Indice 3 : Calculer : roas min/moy/max, roas_j3 moy, variation_ctr moy, duree moy, % sous-perf


In [ ]:
%%sql df_ntile_canal <<
-- 📝 TODO : Composition des tiers par canal
-- 💡 Indice 1 : SUM(CASE WHEN tier=1 THEN 1 ELSE 0 END) AS top_performer
-- 💡 Indice 2 : ROUND(top_performer*100.0/COUNT(*),1) AS pct_top
-- 💡 Indice 3 : GROUP BY canal


### 🧠 Tes observations

> - Le Tier 3 (À optimiser) a-t-il un ROAS J3 déjà plus bas que les autres tiers ? De combien ?
> - Quel canal a la plus forte proportion de Top Performers ?


---
## 7. Heatmap ROAS : jour × canal

### MÉTHODE
SQL pour agréger ROAS et CTR par canal × jour de semaine. Puis `pandas.pivot_table()` pour construire la matrice 4×7 pour la heatmap.

In [ ]:
%%sql df_heatmap_raw <<
-- 📝 TODO : ROAS et CTR moyen par canal × jour de semaine depuis performances_daily (données propres)
-- 💡 Indice 1 : strftime(date_perf, '%w') retourne 0=Dim, 1=Lun... (DuckDB)
-- 💡 Indice 2 : CASE CAST(strftime(date_perf,'%w') AS INTEGER) WHEN 1 THEN '01-Lun' ...
-- 💡 Indice 3 : Filtre is_clean : clics<=impressions AND revenu>=0 AND ctr<=0.20
-- 💡 Indice 4 : AVG(roas_recalc), AVG(ctr_recalc), COUNT(*) AS nb_obs


In [ ]:
# 📝 TODO : Construire la heatmap avec pivot_table + imshow
# 💡 Indice 1 : df_hm = df_heatmap_raw.pivot_table(index='canal', columns='jour_semaine', values='roas_moyen')
# 💡 Indice 2 : ax.imshow(df_hm.values, aspect='auto', cmap='RdYlGn', vmin=1.5, vmax=6.0)
# 💡 Indice 3 : Annoter chaque cellule avec la valeur ROAS

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
# [Heatmap ROAS]
# [Heatmap CTR]
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}marketpulse_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


### 🧠 Tes observations

> - Quel jour de semaine est le plus performant pour Facebook ? Pour Email ?
> - Y a-t-il un pattern weekend systématique ? Qu'en penses-tu pour les campagnes B2B vs B2C ?


---
## 8. LTV par canal d'acquisition

### MÉTHODE
`contacts_crm` contient les clients acquis. LTV = valeur long terme vs ROAS immédiat. Ce tableau peut renverser le classement des canaux.

In [ ]:
%%sql df_ltv_canal <<
-- 📝 TODO : LTV moyenne, médiane, taux actifs et RANK() depuis contacts_crm
-- 💡 Indice 1 : COUNT(*), AVG(ltv_fcfa), PERCENTILE_CONT(0.5) AS ltv_mediane
-- 💡 Indice 2 : SUM(CASE WHEN statut_crm='Actif' THEN 1 ELSE 0 END)*100.0/COUNT(*) AS pct_actifs
-- 💡 Indice 3 : SUM(ltv_fcfa)/NULLIF(SUM(nb_achats),0) AS panier_moyen
-- 💡 Indice 4 : RANK() OVER (ORDER BY AVG(ltv_fcfa) DESC)


### 🧠 Tes observations

> - Le classement LTV est-il identique au classement ROAS du NB3 R2 ? Quel canal monte dans le classement ?
> - Google a un CPA élevé mais quelle est sa LTV ? Le ratio LTV/CPA est-il meilleur que SMS ?


---
## 9. Clients en risque — LAG() PARTITION BY

### MÉTHODE
`LAG() OVER (PARTITION BY client_id ORDER BY mois)` crée une fenêtre par client. La variation moyenne mesure si les performances se dégradent chroniquement.

In [ ]:
%%sql df_clients_risque <<
-- 📝 TODO : Détecter les clients avec dégradation ROAS sur plusieurs mois consécutifs
-- 💡 Indice 1 : WITH camp_mensuel AS (SELECT client_id, strftime(date_debut,'%Y-%m'), AVG(roas_total) GROUP BY...)
-- 💡 Indice 2 : LAG(roas_mois) OVER (PARTITION BY client_id ORDER BY mois) AS roas_prev
-- 💡 Indice 3 : ROUND(roas_mois - LAG(...)) AS delta_roas
-- 💡 Indice 4 : Score risque : CASE WHEN delta_roas_moy < -0.10 AND roas_dernier < 2.0 THEN 'Critique'...


### 🧠 Tes observations

> - Combien de clients sont en statut 'Critique' ? Quels secteurs sont les plus représentés ?
> - Quelle action concrete recommanderais-tu à l'account manager d'un client en statut Critique ?


---
## 10. Coût de la sous-performance

### MÉTHODE
Calcule le ROI du système ML en estimant le budget gaspillé sur les campagnes sous-performantes.

In [ ]:
%%sql df_cout_sousperf <<
-- 📝 TODO : Budget gaspillé sur campagnes sous-perf + économie potentielle ML par client
-- 💡 Indice 1 : SUM(CASE WHEN campagne_sous_performe=1 THEN budget_depense_fcfa ELSE 0 END) AS budget_sous_perf
-- 💡 Indice 2 : Économie potentielle = budget_sous_perf * 0.70 (hypothèse : 70% du budget récupérable à J3)
-- 💡 Indice 3 : RANK() OVER (ORDER BY budget_sous_perf DESC)


### 🧠 Tes observations

> - Quel est le budget total gaspillé sur 24 mois pour les campagnes sous-performantes ?
> - Calcule le ROI du NB5 : si le modèle détecte 75% des sous-performances à J3, combien économise-t-on par an ?


---
## 11. Visualisation synthèse

In [ ]:
# 📝 TODO : Créer une figure 2×2 avec :
#   ax1 (haut-gauche) : barres ROAS/CTR normalisés par canal (Panel comparatif)
#   ax2 (haut-droite) : stacked bars NTILE par canal
#   ax3 (bas-gauche)  : courbe ROAS mensuel + zones de déclin (LAG)
#   ax4 (bas-droite)  : barres LTV par canal + ligne % actifs

# 💡 Indice : fig, axes = plt.subplots(2,2,figsize=(16,12))
# 💡 Indice : Utiliser df_rank_canal, df_ntile_canal, df_lag_mensuel, df_ltv_canal

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
# [Ton code ici]
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}marketpulse_sql_analytics.png', dpi=150)
plt.show()


---
## Bilan NB3

### Insights à documenter — à compléter

| Requête | Pattern SQL | Résultat clé |
|---|---|---|
| R1 KPIs globaux | Agrégation | ROAS=___× · CTR=___% · Sous-perf=___% |
| R2 RANK canaux | `RANK()×4` | Canal #1 : ___ · Canal #4 : ___ |
| R3 Canal×Objectif | `RANK PARTITION` | Conversion → ___ · Notoriété → ___ |
| R4 LAG mensuel | `LAG()` | Nb mois déclin : ___ |
| R5 NTILE | `NTILE(3)` | ROAS J3 moyen Tier 1 : ___ · Tier 3 : ___ |
| R6 Heatmap | `pivot_table` | Meilleur jour/canal : ___ |
| R7 LTV | `RANK` contacts | Canal meilleure LTV : ___ |
| R8 Churn | `LAG PARTITION` | Nb clients Critique : ___ |


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.